In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tomotok.geometry import sparse_line, RegularGrid, generate_sightlines
from tomotok.inversions import PearsonSelector, Tikhonov, GevAlgebraic, SvdAlgebraic, FastSelector
from tomotok.regularisations import all_direction_derivative_matrices, weighted_squares

from tomotok.tools.phantoms import elliptical_flux, gaussian_on_flux


In [ ]:
grid = RegularGrid(100, 100, (.2, .7), (-.5, .5))

## Preparations

### Artificial emissivity

In [ ]:
flux = elliptical_flux(grid.nr, grid.nz, span=1.3)  # artificial flux values
# particularly challenging hollow phantom
phantom = gaussian_on_flux(flux, center=0.3, amplitude=100, limit_width=0.2)

In [ ]:
plt.figure()
plt.imshow(phantom, extent=grid.extent, origin='lower')
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.show()

### Detector setup and geometry matrix

In [ ]:
num = 20
s1, e1 = generate_sightlines(num=(num, 1), fov=(70, 0), pinhole=(1, 0, 0), axis=(-1, 0, 0))
s2, e2 = generate_sightlines(num=(num, 1), fov=(35, 0), pinhole=(.4, 0, .8), axis=(.1, 0, -1), length=1.5)
s3, e3 = generate_sightlines(num=(num, 1), fov=(50, 0), pinhole=(.75, 0, -.4), axis=(-1, 0, 1), length=1.5)
s4, e4 = generate_sightlines(num=(num, 1), fov=(50, 0), pinhole=(.75, 0, .4), axis=(-1, 0, -1), length=1.5)
s = np.concatenate((s1, s2, s3, s4))
e = np.concatenate((e1, e2, e3, e4))

In [ ]:
gmat = sparse_line(s, e, grid, rmin=.2)

In [ ]:
plt.figure()
plt.imshow(gmat.sum(0).reshape(grid.shape), extent=grid.extent, origin='lower')
plt.colorbar(label='Length [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.show()

### Artificial signal

In [ ]:
np.random.seed(20250506)
fwd = gmat @ phantom.flatten()
noise = 0.05 * fwd.max()
signal = fwd
# signal += np.random.normal(0, noise, fwd.shape)

In [ ]:
plt.figure()
plt.plot(fwd, '+', label='forward')
plt.plot(signal, '+', label='noisy')
plt.xlabel('Channel [-]')
plt.ylabel('Signal [-]')
plt.legend()
plt.show()

In [ ]:
errors = (signal + signal.max() ) / 2 * .02
# errors = .001

### Derivative matrices

In [ ]:
# Edge compensations (may) cause regularisation matrices not to be positive definite
# positive definiteness is required for the algebraic algorithms to work correctly
dmats = all_direction_derivative_matrices(grid, compensate_edges=False)
reg = weighted_squares(dmats)

## Inversions

In [ ]:
select = PearsonSelector(bounds=(-10, -4), iter_max=20, tolerance=0.001)
chol = Tikhonov(regularisation_selector=select)

out_chol, stats_chol = chol(signal, gmat, reg, errors)

In [ ]:
print(stats_chol)

In [ ]:
plt.figure()
plt.imshow(out_chol.reshape(grid.shape), extent=grid.extent, origin='lower')
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.title('Tikhonov')

plt.figure()
plt.plot(signal, label='signal')
plt.plot(gmat @ out_chol, label='retrofit')
plt.show()

### Generalised Eigenvalue (GEV)

#### Fast regularisation

In [ ]:
selector = FastSelector(method='median')
gev = GevAlgebraic(regularisation_selector=selector)  # Uses methods from scipy.sparse

In [ ]:
out_gev, stats_gev = gev(signal, gmat, reg, errors)
print(stats_gev)

In [ ]:
plt.figure()
plt.title('Fast GEV')
plt.imshow(out_gev.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.show()

In [ ]:
plt.figure()
plt.plot(signal, label='Artificial signal')
plt.plot(gmat @ out_gev, label='Retrofit')
plt.xlabel('Channel [-]')
plt.ylabel('Signal [-]')
plt.legend()
plt.show()

#### Pearson $\chi^2$ test

In [ ]:
pselector = PearsonSelector(bounds=(-30, -1), iter_max=20, tolerance=0.001)
pgev = GevAlgebraic(regularisation_selector=pselector)
out_pgev, stats_pgev = pgev(signal, gmat, reg, errors)
print(stats_pgev)

In [ ]:
plt.figure()
plt.title('Pearson GEV')
plt.imshow(out_pgev.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')

plt.figure()
plt.plot(signal, label='Artificial signal')
plt.plot(gmat @ out_pgev, label='Retrofit')
plt.xlabel('Channel [-]')
plt.ylabel('Signal [-]')
plt.legend()

plt.show()

### Singular Value Decomposition

#### Fast regularisation

In [ ]:
svd = SvdAlgebraic(regularisation_selector=selector)  # no sparse optimization

In [ ]:
out_svd, stats_svd = svd(signal, gmat, reg, errors)
print(stats_svd)

In [ ]:
plt.figure()
plt.title('Fast SVD')
plt.imshow(out_svd.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')

plt.figure()
plt.plot(signal)
plt.plot(gmat @ out_svd)
plt.show()

#### Pearson $\chi^2$

In [ ]:
psvd = SvdAlgebraic(regularisation_selector=pselector)
out_psvd, stats_psvd = psvd(signal, gmat, reg, errors)
print(stats_psvd)

In [ ]:
plt.figure()
plt.title('Pearson SVD')
plt.imshow(out_psvd.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')

plt.figure()
plt.plot(signal, label='Artificial signal')
plt.plot(gmat @ out_psvd, label='Retrofit')
plt.xlabel('Channel [-]')
plt.ylabel('Signal [-]')
plt.legend()
plt.show()